In [1]:
# !pip install langchain-text-splitters
#! pip install chromadb
# ! pip install pdfplumber
# ! pip install fitz
# ! pip install PyMuPDF 

In [1]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm
from transformers import pipeline
import pdfplumber
from langchain_core.prompts import ChatPromptTemplate
import json
import os
from chromadb import Client
from chromadb.utils import embedding_functions
from chromadb.config import Settings
import fitz  # PyMuPDF

## PDF to Text

In [2]:
PDF_PATH = "D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\Business_Review.txt" 
TARGET_CHUNK_TOKENS = 300 # ~225 words
OVERLAP_TOKENS = 50
EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2')
# LLM = None # Lazy load later for agentic/contextual
LLM = pipeline("text-generation", model="microsoft/Phi-3-mini-4k-instruct",
                       device="cpu", dtype="auto", trust_remote_code=False)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


In [3]:
def load_pdf(path: str) -> str:
    """Extract text from PDF. Uses PyMuPDF, falls back to pdfplumber for tables."""
    try:
        doc = fitz.open(path)
        pages = [page.get_text("text") for page in doc]
        doc.close()
        text = "\n\n".join(pages)
    except:
        with pdfplumber.open(path) as pdf:
            pages = [p.extract_text() or "" for p in pdf.pages]
            text = "\n\n".join(pages)

    # Clean common PDF artifacts
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r'--- Page \d+ ---', '', text)
    return text.strip()

# Different Chunking Strategies

## Fixed Length Chuking

In [4]:
def fixed_length_chunking(text: str, chunk_size=TARGET_CHUNK_TOKENS, overlap=OVERLAP_TOKENS):
    words = text.split()
    chunks, step = [], chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip(): chunks.append(chunk)
    return chunks

## Content Aware Chunking

In [5]:
def content_aware_chunking(text: str, min_words=150, max_words=450):
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for para in paras:
        if len((current + " " + para).split()) <= max_words:
            current = (current + "\n\n" + para).strip()
        else:
            if len(current.split()) >= min_words: chunks.append(current)
            current = para
    if current and len(current.split()) >= min_words: chunks.append(current)
    elif current and chunks: chunks[-1] += "\n\n" + current
    return chunks

## Recursive Chunking

In [6]:
def recursive_chunking(text: str, chunk_size=TARGET_CHUNK_TOKENS, overlap=OVERLAP_TOKENS):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=int(chunk_size * 4.5), # ~4.5 chars per token
        chunk_overlap=int(overlap * 4.5),
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return splitter.split_text(text)

## Structure based chunking

In [7]:
def structure_based_chunking(text: str, min_words=100):
    pattern = r'\n(?=# |## |Chapter |Section |^\d+\.\s+[A-Z])'
    sections = re.split(pattern, text, flags=re.MULTILINE)
    chunks, buffer = [], ""
    for sec in sections:
        sec = sec.strip()
        if not sec: continue
        if len((buffer + " " + sec).split()) < min_words:
            buffer = (buffer + "\n\n" + sec).strip()
        else:
            if buffer: chunks.append(buffer)
            buffer = sec
    if buffer: chunks.append(buffer)
    return chunks if chunks else [text]

## Semantic Chunking

In [8]:
def semantic_chunk(text: str, percentile_threshold=95):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sentences) < 3: return [text]
    embeddings = EMBED_MODEL.encode(sentences, show_progress_bar=False)
    sims = [cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0] for i in range(len(embeddings)-1)]
    threshold = np.percentile(sims, 100 - percentile_threshold)
    breaks = [i for i, s in enumerate(sims) if s < threshold]
    chunks, start = [], 0
    for b in breaks:
        chunks.append(' '.join(sentences[start:b+1]))
        start = b + 1
    chunks.append(' '.join(sentences[start:]))
    # Merge tiny chunks
    merged = []
    for c in chunks:
        if merged and len(c.split()) < 50: merged[-1] += " " + c
        else: merged.append(c)
    return merged

## Sliding Window Chunking

In [9]:
def sliding_window_chunking(text: str, window=TARGET_CHUNK_TOKENS, stride=150):
    words = text.split()
    chunks = []
    for i in range(0, len(words), stride):
        chunk = ' '.join(words[i:i + window])
        if len(chunk.split()) > 50: chunks.append(chunk)
        if i + window >= len(words): break
    return chunks

## Proposition Chunking

In [10]:
def proposition_chunking(text: str):
    global LLM
    if LLM is None:
        LLM = pipeline("text-generation", model="microsoft/Phi-3-mini-4k-instruct",
                       device="cpu", torch_dtype="auto", trust_remote_code=True)

    # Do it in 2k char batches to avoid context limit
    batch_size = 2000
    batches = [text[i:i+batch_size] for i in range(0, len(text), batch_size)]
    all_props = []
    for batch in tqdm(batches, desc="Proposition chunking"):
        prompt = f"Break the following into simple, self-contained facts. One fact per line:\n\n{batch}\n\nFacts:"
        resp = LLM(prompt, max_new_tokens=512, do_sample=False, return_full_text=False)
        props = [p.strip() for p in resp[0]['generated_text'].split("\n") if p.strip()]
        all_props.extend(props)
    return all_props

## Parent Child Chunking 

In [11]:
def parent_child_chunking(text: str, child_size=200, parent_size=1500):
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=parent_size*4, chunk_overlap=200)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=child_size*4, chunk_overlap=50)
    parents = parent_splitter.split_text(text)
    children = []
    parent_map = []
    for i, p in enumerate(parents):
        c = child_splitter.split_text(p)
        children.extend(c)
        parent_map.extend([i] * len(c))
    return {"children": children, "parents": parents, "parent_map": parent_map}


## Agentic chunking

In [12]:
def agentic_chunking(text: str):
    global LLM
    if LLM is None:
        LLM = pipeline("text-generation", model="microsoft/Phi-3-mini-4k-instruct",
                       device="cpu", torch_dtype="auto", trust_remote_code=True)
    prompt = f"""Split this document into semantically coherent chunks of 150-350 words.
Each chunk must be self-contained. Return as JSON list of strings.
Document: {text[:12000]}
JSON:""" # Truncate for context limit
    resp = LLM(prompt, max_new_tokens=2048, do_sample=False, return_full_text=False)
    try:
        return json.loads(resp[0]['generated_text'])
    except:
        return recursive_chunking(text) # fallback

## Late Chunking

In [13]:
def late_chunking(text: str, chunk_tokens=256):
    from transformers import AutoModel, AutoTokenizer
    model = AutoModel.from_pretrained('jinaai/jina-embeddings-v3', trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained('jinaai/jina-embeddings-v3', trust_remote_code=True)

    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=8192)
    embeddings = model(**inputs).last_hidden_state[0].detach().numpy()

    chunks = [embeddings[i:i+chunk_tokens] for i in range(0, len(embeddings), chunk_tokens)]
    return chunks # Note: these are embeddings, not text

## Hierarchial Chunking

In [14]:
def hierarchical_chunking(text: str):
    # Level 1: big chunks
    l1 = recursive_chunking(text, chunk_size=1500, overlap=100)
    # Level 2: summarize each L1, then chunk summaries
    global LLM
    if LLM is None:
        LLM = pipeline("text-generation", model="microsoft/Phi-3-mini-4k-instruct",
                       device="cpu", torch_dtype="auto", trust_remote_code=True)
    summaries = []
    for chunk in tqdm(l1, desc="Hierarchical L1->L2"):
        prompt = f"Summarize in 2 sentences: {chunk[:2000]}"
        resp = LLM(prompt, max_new_tokens=100, do_sample=False, return_full_text=False)
        summaries.append(resp[0]['generated_text'])
    l2_text = "\n\n".join(summaries)
    l2 = recursive_chunking(l2_text, chunk_size=300, overlap=50)
    return {"L1": l1, "L2": l2}

## Contextual Chunking

In [15]:
PRONOUN_PAT = re.compile(r'\b(it|this|that|these|those|he|she|they|his|her|their|its|we|our)\b', re.I)
def contextual_chunking(text: str):
    base = semantic_chunk(text, percentile_threshold=95)
    global LLM
    if LLM is None:
        LLM = pipeline("text-generation", model="microsoft/Phi-3-mini-4k-instruct",
                       device="cpu", torch_dtype="auto", trust_remote_code=True)

    out = [base[0]]
    for i in range(1, len(base)):
        curr, prev = base[i], out[-1]
        first_sent = re.split(r'[.!?]', curr.strip())[0]
        if len(curr.split()) < 150 and PRONOUN_PAT.search(first_sent):
            prompt = f"Previous: {prev[-500:]}\nCurrent: {curr[:300]}\nIf current uses pronouns from previous, write 1-sentence context. Else NONE:"
            resp = LLM(prompt, max_new_tokens=60, do_sample=False, return_full_text=False)
            ctx = resp[0]['generated_text'].strip()
            if not ctx.upper().startswith("NONE"):
                out.append(f"[Context: {ctx}] {curr}")
            else:
                out.append(curr)
        else:
            out.append(curr)
    return out

# EVALUATION 

In [16]:
def calc_metrics(chunks, text):
    if not chunks: return {}

    # 1. Efficiency
    lens = [len(c.split()) for c in chunks]
    efficiency = {"avg_tokens": np.mean(lens), "std": np.std(lens), "count": len(chunks)}

    # 2. Semantic Coherence
    if len(chunks) < 2:
        coherence = 1.0
    else:
        embs = EMBED_MODEL.encode(chunks, show_progress_bar=False)
        sims = [cosine_similarity([embs[i]], [embs[i+1]])[0][0] for i in range(len(embs)-1)]
        coherence = np.mean(sims)

    # 3. Context Completeness
    bad_starts = sum(1 for c in chunks if PRONOUN_PAT.match(c.strip()))
    completeness = 1 - (bad_starts / len(chunks))

    # 4. Retrieval Quality - use simple keyword queries from doc
    queries = [
        " ".join(text.split()[:5]), # first 5 words
        " ".join(text.split()[len(text.split())//2:len(text.split())//2+5]), # middle
    ]
    chunk_embs = EMBED_MODEL.encode(chunks, show_progress_bar=False)
    query_embs = EMBED_MODEL.encode(queries, show_progress_bar=False)
    scores = []
    for q_emb in query_embs:
        sims = cosine_similarity([q_emb], chunk_embs)[0]
        scores.append(np.max(sims)) # best match score
    retrieval = np.mean(scores)

    return {
        "Semantic Coherence": round(coherence, 3),
        "Context Completeness": round(completeness, 3),
        "Retrieval Quality": round(retrieval, 3),
        "Efficiency": f"{efficiency['avg_tokens']:.1f} avg tokens, std {efficiency['std']:.2f}",
        "Chunks": efficiency['count']
    }

In [19]:
def run_all_strategies(pdf_path: str):
    print(f"Loading PDF: {pdf_path}")
    text = load_pdf(pdf_path)
    print(f"Loaded {len(text.split())} words")

    strategies = {
        "Fixed Length": fixed_length_chunking,
        "Content Aware": content_aware_chunking,
        "Recursive": recursive_chunking,
        "Structure Based": structure_based_chunking,
        "Semantic": semantic_chunk,
        "Contextual": contextual_chunking,
        "Sliding Window": sliding_window_chunking,
        "Proposition": proposition_chunking,
        "Parent-Child": parent_child_chunking,
        "Agentic": agentic_chunking,
        "Hierarchical": hierarchical_chunking,
        # "Late Chunking": late_chunking, # returns embeddings, skip for text eval
    }

    results = {}
    for name, func in strategies.items():
        print(f"\n=== {name} ===")
        try:
            out = func(text)
            # Handle special returns
            if name == "Parent-Child": chunks = out["children"]
            elif name == "Hierarchical": chunks = out["L2"]
            else: chunks = out

            metrics = calc_metrics(chunks, text)
            results[name] = metrics

            for k, v in metrics.items():
                print(f" {k}: {v}")
            print("-" * 60)
        except Exception as e:
            print(f" ERROR: {e}")
            results[name] = {"error": str(e)}
        # finally:
        #     return results
    return results
    # # Save results
    # with open("chunking_results.json", "w") as f:
    #     json.dump(results, f, indent=2)
    # print("\nSaved to chunking_results.json")
    # return results

In [ ]:
path = PDF_PATH
results = run_all_strategies(path)
# Save results
# with open("chunking_results.json", "w") as f:
#     json.dump(results, f, indent=2)
# print("\nSaved to chunking_results.json")


Loading PDF: D:\LLM-Learning-Journey\LLM-Learning-Journey\Week3 - Embedding and vector Database\Data\Business_Review.txt
Loaded 372 words

=== Fixed Length ===
 Semantic Coherence: 0.45500001311302185
 Context Completeness: 1.0
 Retrieval Quality: 0.4350000023841858
 Efficiency: 211.0 avg tokens, std 89.00
 Chunks: 2
------------------------------------------------------------

=== Content Aware ===
 Semantic Coherence: 1.0
 Context Completeness: 1.0
 Retrieval Quality: 0.4350000023841858
 Efficiency: 372.0 avg tokens, std 0.00
 Chunks: 1
------------------------------------------------------------

=== Recursive ===
 Semantic Coherence: 0.49900001287460327
 Context Completeness: 1.0
 Retrieval Quality: 0.4350000023841858
 Efficiency: 186.0 avg tokens, std 23.00
 Chunks: 2
------------------------------------------------------------

=== Structure Based ===
 Semantic Coherence: 0.4429999887943268
 Context Completeness: 1.0
 Retrieval Quality: 0.5540000200271606
 Efficiency: 74.4 avg to

Proposition chunking: 100%|██████████| 2/2 [06:08<00:00, 184.29s/it]


 Semantic Coherence: 0.28200000524520874
 Context Completeness: 1.0
 Retrieval Quality: 0.4959999918937683
 Efficiency: 10.1 avg tokens, std 16.52
 Chunks: 62
------------------------------------------------------------

=== Parent-Child ===
 Semantic Coherence: 0.49000000953674316
 Context Completeness: 1.0
 Retrieval Quality: 0.4309999942779541
 Efficiency: 95.5 avg tokens, std 42.43
 Chunks: 4
------------------------------------------------------------

=== Agentic ===


In [20]:
results

{'Fixed Length': {'Semantic Coherence': np.float32(0.455),
  'Context Completeness': 1.0,
  'Retrieval Quality': np.float32(0.435),
  'Efficiency': '211.0 avg tokens, std 89.00',
  'Chunks': 2}}